<a href="https://colab.research.google.com/github/XanderPatterson/4487-IS-Class-Labs/blob/main/assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [3]:
import pandas as pd

df = pd.read_csv("airbnb_listings.csv")

display(df.head())
df.info()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 77 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            459 non-null    int64  
 1   listing_url                                   459 non-null    object 
 2   scrape_id                                     459 non-null    int64  
 3   last_scraped                                  459 non-null    object 
 4   source                                        459 non-null    object 
 5   name                                          459 non-null    object 
 6   description                                   449 non-null    object 
 7   neighborhood_overview                         196 non-null    object 
 8   host_url                                      459 non-null    object 
 9   host_name                                     459 non-null    obj

### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. It includes details about Airbnb properties, such as host information, room types, pricing, locations, number of bedrooms/bathrooms, and guest reviews.

2. I used the back up data set from our class files and in the data set there are 459 rows known as listings and 77 columns known as features.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [15]:
columns_to_drop = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
    'description', 'neighborhood_overview', 'host_url', 'host_name',
    'host_since', 'host_location', 'host_about', 'host_thumbnail_url',
    'host_picture_url', 'host_neighbourhood', 'host_verifications',
    'amenities', 'calendar_updated', 'calendar_last_scraped', 'first_review',
    'last_review', 'license', 'bathrooms_text', 'property_type', 'room_type',
    'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'host_response_time', 'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'has_availability']

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("New shape:", df.shape)
display(df.head())

New shape: (459, 41)


,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,70.0,28,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,104.0,2,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,75.0,2,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,101.0,2,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,110.0,1,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped ID numbers, URLs, text descriptions, and categorical columns. I did this because linear regression requires numbers in the data to calculate coefficients and unique IDs and things like that don't help predict a price.

2. Including text descriptions or unique IDs would cause errors, add random noise, or cause the model to memorize individual listings rather than learning general pricing patterns.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [5]:
correlation_matrix = df.corr()

price_correlations = correlation_matrix['price'].sort_values(ascending=False)

print(price_correlations)

price                                           1.000000
accommodates                                    0.579588
beds                                            0.547032
bedrooms                                        0.499286
bathrooms                                       0.468030
estimated_revenue_l365d                         0.249488
maximum_maximum_nights                          0.122872
minimum_maximum_nights                          0.112166
maximum_nights_avg_ntm                          0.111271
availability_30                                 0.108409
availability_60                                 0.060509
availability_90                                 0.040997
calculated_host_listings_count_entire_homes     0.033206
availability_eoy                                0.032537
review_scores_value                             0.018269
maximum_nights                                  0.017992
calculated_host_listings_count                  0.015773
review_scores_cleanliness      

### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. Accommodates, beds, and bedrooms had the strongest positive correlation with price. Review scores communication and longitude had the strongest negative correlations.

2. Capacity variables like accommodates, beds, and bedrooms will likely be the most useful predictors since larger properties cost more. These seem to be the stronger parts in what drives the coeficients to be stronger

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [9]:
y = df['price']

X = df.drop(columns=['price']).select_dtypes(include=['number']).fillna(0)

print("Features shape (X):", X.shape)
print("Target shape (y):", y.shape)

Features shape (X): (459, 40)
Target shape (y): (459,)


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I am using all the remaining numeric features in the dataset like accommodates, beds, bedrooms, and bathrooms to make my  predictions.

2. This is a regression problem because we are predicting a continuous numerical value this being specific dollar amount for the price rather than sorting the listings into distinct categories or labels like "expensive" vs. "cheap". This helps get a more interactive representation of the data.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (367, 40)
X_test shape: (92, 40)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [11]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model successfully trained and predictions generated!")

Model successfully trained and predictions generated!


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [12]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 9845.36
R² Score: -0.7542


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. The R² score is around -0.75. This score means our current model performs worse than a horizontal average line, indicating that including all numeric features with missing values filled with zeros created too much noise. Meaning we could've gotten more accurate data.

2. The Mean Squared Error is 9,845.36, which is pretty large. We can improve it moving forward by filtering down to only the most impactful predictors like accommodates, beds, and bedrooms instead of using every single numeric column.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [13]:
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_})

coefficients_df['Absolute_Coefficient'] = coefficients_df['Coefficient'].abs()
coefficients_df = coefficients_df.sort_values(by='Absolute_Coefficient', ascending=False).drop(columns=['Absolute_Coefficient'])

display(coefficients_df)

,Feature,Coefficient
2,latitude,-747.983973
3,longitude,-224.362487
33,review_scores_value,164.918907
31,review_scores_communication,-114.664807
27,review_scores_rating,-72.735765
30,review_scores_checkin,-72.505006
28,review_scores_accuracy,39.890916
11,maximum_minimum_nights,-39.092536
14,minimum_nights_avg_ntm,36.902999
5,bathrooms,23.158276


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. Geographical coordinates like latitude and review-related ratings such as review_scores_value showed the largest coefficient magnitudes in this type of model.

2. Yes, latitude and certain review scores had large negative coefficients, the model balances out competing variables or because the data wasn't standardized, which can cause weights to shift in ways that don't match simple logic.
3. Location and specific guest satisfaction metrics were ones that influenced how pricing coefficients are weighted by the linear regression.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [14]:
top_features = ['accommodates', 'beds', 'bedrooms', 'bathrooms']

X_refined = X[top_features].fillna(0)

X_train_ref, X_test_ref, y_train_ref, y_test_ref = train_test_split(X_refined, y, test_size=0.2, random_state=42)

refined_model = LinearRegression()
refined_model.fit(X_train_ref, y_train_ref)

y_pred_ref = refined_model.predict(X_test_ref)

mse_ref = mean_squared_error(y_test_ref, y_pred_ref)
r2_ref = r2_score(y_test_ref, y_pred_ref)

print(f"Refined Mean Squared Error (MSE): {mse_ref:.2f}")
print(f"Refined R² Score: {r2_ref:.4f}")

Refined Mean Squared Error (MSE): 3416.04
Refined R² Score: 0.3914


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept capacity features like accommodates, beds, bedrooms, and bathrooms because they had the strongest logical and statistical correlation with price.

2. Yes, the performance improved because removing noisy variables and irrelevant features allowed the model to focus on the strongest price drivers.

3. I would recommend the refined model because it is simpler, more accurate, and easier to explain to hosts and business partners. Removing the unneccessary data helps prevent an overload of data. Sometimes you are just trying to syphin through too much.

4. It demonstrates practical data cleaning and feature selection skills to build a functional predictive model for business decision-making. It is crazy to me how you can take some much data and then find ways to clean it and make it actually usefull, it is something that is amazing me about the final project that we are doing in class.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. It helped answer what physical property features drive listing prices up or down. By taking the coefficents of the data and finding out what drove it more help me focus on the most important.

2. I would recommend that hosts focus heavily on maximizing their property's capacity like adding beds or comfortable accommodation space since it has the most direct impact on pricing.

3. We could add categorical variables like room type or neighborhood using one-hot encoding or handle missing data using imputation instead of just filling with zeros. There could still be some relevance in this type of categorical data.

4. It ties directly into applying data science and regression techniques to solve practical, real-world business pricing strategies. When preparing a project or trying to get through a lot of unwanted or un useful data, this provides a way to clean that data, find out what is driving it and then what is making it useful. This helps to get accuracy and the most important parts in the data.  

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"